# Pesquisa Avançada: Detecção de Câncer via Feature Engineering Progressiva
**Objetivo:** Medir o ganho incremental de Cor, Textura e Espectro na classificação de lesões de pele.

---

In [ ]:
!pip install datasets tensorflow pandas matplotlib seaborn scikit-learn scikit-image -q
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from skimage.color import rgb2lab, rgb2hsv
from skimage.measure import moments, moments_hu
from skimage.filters import threshold_otsu
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
import gc

print("--- Configurando Acelerador ---")
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver(tpu='local')
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print('Hardware: TPU Ativa!')
except:
    strategy = tf.distribute.get_strategy()
    print('Hardware: GPU/CPU')

IMG_SIZE = (224, 224)
BATCH_SIZE = 16 * strategy.num_replicas_in_sync

## 1. Carregamento e Balanceamento (Máx 4k Nevi)
Usamos streaming para garantir que o download não trave no Kaggle.

In [ ]:
print("--- Iniciando Streaming Híbrido ---")
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except: hf_token = None

ds_stream = load_dataset("marmal88/skin_cancer", split='train', streaming=True, token=hf_token)

limit_nevi = 4000
counts = {}
X_raw, y_labels = [], []

for example in ds_stream:
    lbl = example['dx']
    counts[lbl] = counts.get(lbl, 0)
    
    # Se for Nevi e já atingiu 4000, pula. Senão, pega tudo.
    if lbl == 'melanocytic_Nevi' and counts[lbl] >= limit_nevi:
        continue
        
    img = example['image'].convert('RGB').resize(IMG_SIZE)
    X_raw.append(np.array(img, dtype='uint8'))
    y_labels.append(lbl)
    counts[lbl] += 1
    if sum(counts.values()) % 1000 == 0: print(f"Capturadas: {sum(counts.values())} | {counts}")

X_raw = np.array(X_raw)
y_labels = np.array(y_labels)
gc.collect()
print(f"Dataset Finalizado: {X_raw.shape}")

## 2. Feature Engineering Progressiva
Criamos extratores para Cor, Textura e Espectro.

In [ ]:
from PIL import Image
print("--- Extraindo Atributos Dermatológicos (ABCD) ---")

def extract_all_features(img_uint8):
    img_f = img_uint8.astype('float32') / 255.0
    img_gray = np.array(Image.fromarray(img_uint8).convert('L'))
    
    # 1. COR (RGB + HSV) - Regra 'C'
    color_rgb = img_f.mean(axis=(0,1))
    img_hsv = rgb2hsv(img_f)
    color_hsv = img_hsv.mean(axis=(0,1))
    color_feat = np.concatenate([color_rgb, color_hsv])
    
    # 2. MORFOLOGIA (GLCM) - Regra 'B' (Bordas/Textura)
    glcm = graycomatrix(img_gray, [1], [0], levels=256, symmetric=True, normed=True)
    texture_feat = [
        graycoprops(glcm, 'contrast')[0,0], 
        graycoprops(glcm, 'homogeneity')[0,0],
        graycoprops(glcm, 'energy')[0,0],
        graycoprops(glcm, 'correlation')[0,0]
    ]
    
    # 3. FORMA (Hu Moments) - Regra 'A' (Assimetria)
    try:
        thresh = threshold_otsu(img_gray)
        binary = (img_gray > thresh).astype(np.uint8)
        if binary[0,0] == 1: binary = 1 - binary
        mu = moments(binary)
        hu = moments_hu(mu)
        # Log transform para estabilidade numérica (escala logarítmica)
        hu = -np.sign(hu) * np.log10(np.abs(hu) + 1e-12)
    except:
        hu = np.zeros(7)
    shape_feat = hu
    
    return color_feat, texture_feat, shape_feat

print("Processando imagens...")
feats = [extract_all_features(X_raw[i]) for i in range(len(X_raw))]
f_color = np.nan_to_num(np.array([f[0] for f in feats]))
f_texture = np.nan_to_num(np.array([f[1] for f in feats]))
f_shape = np.nan_to_num(np.array([f[2] for f in feats]))

X_final = np.hstack([f_color, f_texture, f_shape])
print(f"Finalizado! Atributos totais extraídos: {X_final.shape[1]}")

## 3. Gráfico de Ganho (Random Forest)
Medindo a importância de cada técnica.

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_enc = le.fit_transform(y_labels)

results = []
def test_rf(X_data, name):
    X_train, X_test, y_train, y_test = train_test_split(X_data, y_enc, test_size=0.2, random_state=42, stratify=y_enc)
    rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    rf.fit(X_train, y_train)
    acc = accuracy_score(y_test, rf.predict(X_test))
    results.append({'Tecnica': name, 'Acuracia': acc})
    return rf

print("Treinando modelos incrementais...")
test_rf(f_color, "A: Cor (RGB+HSV)")
test_rf(np.hstack([f_color, f_texture]), "B: Cor + Textura (GLCM)")
rf_final = test_rf(X_final, "C: Cor + Tex + Forma (Hu)")

df_res = pd.DataFrame(results)
plt.figure(figsize=(10,5))
sns.lineplot(data=df_res, x='Tecnica', y='Acuracia', marker='o', linewidth=3)
plt.title("Gráfico de Ganho por Técnica (Random Forest)")
plt.grid(True)
plt.show()

## 4. Deep Learning: Viabilidade de Alta Fidelidade
Modelo Dual-Input para validar o ganho com redes neurais.

In [ ]:
def dl_pipeline(img_uint8, label):
    def process(img):
        img_f = img.astype('float32') / 255.0
        img_lab = rgb2lab(img_f)
        # Canal a* normalizado para vascularização
        a_ch = (img_lab[:,:,1] + 128) / 255.0
        return img_f, a_ch[..., np.newaxis].astype('float32')
    
    rgb, spec = tf.numpy_function(process, [img_uint8], [tf.float32, tf.float32])
    rgb.set_shape((224, 224, 3))
    spec.set_shape((224, 224, 1))
    return {"rgb_input": rgb, "spec_input": spec}, label

X_tr, X_ts, y_tr, y_ts = train_test_split(X_raw, y_enc, test_size=0.15, stratify=y_enc)
train_ds = tf.data.Dataset.from_tensor_slices((X_tr, y_tr)).shuffle(1000).map(dl_pipeline).batch(BATCH_SIZE).prefetch(2)
test_ds = tf.data.Dataset.from_tensor_slices((X_ts, y_ts)).map(dl_pipeline).batch(BATCH_SIZE).prefetch(2)

with strategy.scope():
    in_rgb = layers.Input(shape=(224, 224, 3), name="rgb_input")
    x1 = tf.keras.applications.EfficientNetV2B0(include_top=False, weights='imagenet')(in_rgb)
    x1 = layers.GlobalAveragePooling2D()(x1)
    
    in_spec = layers.Input(shape=(224, 224, 1), name="spec_input")
    x2 = layers.Conv2D(16, 3, activation='relu')(in_spec)
    x2 = layers.GlobalAveragePooling2D()(x2)
    
    comb = layers.Concatenate()([x1, x2])
    out = layers.Dense(len(le.classes_), activation='softmax')(comb)
    
    model = models.Model(inputs=[in_rgb, in_spec], outputs=out)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("Treinando Deep Learning...")
model.fit(train_ds, validation_data=test_ds, epochs=10)

y_pred = np.argmax(model.predict(test_ds), axis=1)
plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix(y_ts, y_pred), annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_, cmap='Blues')
plt.title("Matriz de Confusão Final (Deep Learning)")
plt.show()